# IDS multiclasse SOME/IP — pipeline ponta a ponta (split temporal)

> **Notebook-artefato do artigo.** Roda o método **do tráfego cru ao resultado**, com **todo o código
> de extração de features embutido e visível** (byte model do Kim, parser SOME/IP, rotulador, e as 16
> features comportamentais). Baixa os PCAPs, **extrai as features**, treina o XGBoost e avalia com
> **split temporal por arquivo** (honesto, *in-scope*). Salva em `results/`: tabela por classe,
> **matriz de confusão**, **curvas ROC/PR**, **importância das features** e **métricas de tempo**.

Classes: `normal, dos, fuzzy, mitm_single, mitm_multi`. XGBoost: 300 árvores, `max_depth=8`, `lr=0.3`.

**Ative a GPU:** Ambiente de execução → Alterar tipo → **T4 GPU** (use **High-RAM** se houver). Depois: **Executar tudo**.

In [ ]:
# Setup: clona o repo (para scripts/download_pcaps.py). O codigo de features esta EMBUTIDO abaixo.
import os
REPO = 'someip-ids-multiclass-contentext'
def in_repo(): return os.path.exists('scripts/download_pcaps.py')
if not in_repo() and os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
if not in_repo() and os.path.isdir(REPO):
    os.chdir(REPO)
if not in_repo():
    os.system('apt-get -qq install -y git-lfs >/dev/null 2>&1; git lfs install')
    os.system(f'GIT_LFS_SKIP_SMUDGE=1 git clone -q https://github.com/GuilhermeFrick/{REPO}.git')
    os.chdir(REPO)
print('cwd:', os.getcwd(), '| repo ok:', in_repo())

In [ ]:
# Maquina onde esta rodando (para reportar o custo de execucao)
import platform, subprocess
MACHINE = {'platform': platform.platform(), 'processor': platform.processor() or platform.machine(),
           'cpu_logical': os.cpu_count()}
try:
    import psutil
    MACHINE['cpu_physical'] = psutil.cpu_count(logical=False); MACHINE['ram_gb'] = round(psutil.virtual_memory().total/1e9, 1)
except Exception:
    MACHINE['cpu_physical'] = None; MACHINE['ram_gb'] = None
try:
    MACHINE['gpu'] = subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'], text=True).strip()
except Exception:
    MACHINE['gpu'] = None
print('Plataforma :', MACHINE['platform']); print('Processador:', MACHINE['processor'])
print(f"CPU cores  : logicos = {MACHINE['cpu_logical']} | fisicos = {MACHINE['cpu_physical']}")
print('RAM        :', MACHINE['ram_gb'], 'GB'); print('GPU        :', MACHINE['gpu'] or 'nenhuma (CPU)')

In [ ]:
# Dependencias e imports comuns usados pelo codigo de features
try:
    import scapy  # noqa: F401
except ImportError:
    os.system('pip install -q scapy')
import socket, time, json
from dataclasses import dataclass, field
from collections import defaultdict, deque
import numpy as np
from scapy.utils import RawPcapReader
print('imports OK')

## Feature engineering — 1/5 · Modelo de bytes posicional (Kim et al., 2026)

Distribuição de bytes por posição, com suavização de Laplace (Eq. 3); fornece a **log-verossimilhança**
(Eq. 5), a **entropia cruzada** (Eq. 6) e a **distância de Hamming** (Eq. 7).

In [ ]:
class ByteModel:
    def __init__(self, L: int, alpha: float = 1.0):
        self.L = int(L)
        self.alpha = float(alpha)
        self._counts = np.zeros((self.L, 256), dtype=np.float64)
        self.logP: np.ndarray | None = None

    # ---- treino ----
    def update(self, payload: bytes) -> None:
        """Acumula contagens posicionais de um payload benigno."""
        k = min(len(payload), self.L)
        if k == 0:
            return
        arr = np.frombuffer(payload[:k], dtype=np.uint8)
        self._counts[np.arange(k), arr] += 1.0

    def finalize(self) -> "ByteModel":
        """Calcula P_i(b) com Laplace (Eq. 3) e guarda log P."""
        P = (self._counts + self.alpha) / (
            self._counts.sum(axis=1, keepdims=True) + self.alpha * 256.0
        )
        self.logP = np.log(P)
        return self

    # ---- inferência ----
    def loglik(self, payload: bytes) -> float:
        """logL(x) somada nas posições disponíveis (Eq. 5). 0.0 se payload vazio."""
        if self.logP is None:
            raise RuntimeError("ByteModel não finalizado")
        k = min(len(payload), self.L)
        if k == 0:
            return 0.0
        arr = np.frombuffer(payload[:k], dtype=np.uint8)
        return float(self.logP[np.arange(k), arr].sum())

    def cross_entropy(self, payload: bytes) -> float:
        """H(x;P) = -(1/L) sum log P_i(x_i) (Eq. 6). 0.0 se payload vazio."""
        if self.logP is None:
            raise RuntimeError("ByteModel não finalizado")
        k = min(len(payload), self.L)
        if k == 0:
            return 0.0
        arr = np.frombuffer(payload[:k], dtype=np.uint8)
        return float(-self.logP[np.arange(k), arr].sum() / k)


def hamming(a: bytes, b: bytes) -> int:
    """Distância de Hamming em nível de byte entre dois payloads (Eq. 7).

    Conta posições com bytes diferentes no comprimento comum + diferença de comprimento
    (bytes presentes em um e ausentes no outro contam como mudança).
    """
    if not a and not b:
        return 0
    n = min(len(a), len(b))
    if n == 0:
        return max(len(a), len(b))
    aa = np.frombuffer(a[:n], dtype=np.uint8)
    bb = np.frombuffer(b[:n], dtype=np.uint8)
    return int(np.count_nonzero(aa != bb) + abs(len(a) - len(b)))

## Feature engineering — 2/5 · Parser SOME/IP / SOME/IP-SD

Faz o parse do frame Ethernet/IP/UDP-TCP cru e expõe os payloads por camada (`pl_l4`, `pl_someip`,
`pl_sd`), o cabeçalho SOME/IP e as entradas de Service Discovery. Define também a **chave de fluxo**.

In [ ]:
SD_PORT = 30490


ETH_IPV4 = b"\x08\x00"


MT_NOTIFICATION = 0x02


SD_FIND = 0x00


SD_OFFER = 0x01


def _ip(b: bytes) -> str:
    return socket.inet_ntoa(b)


@dataclass(slots=True)
class Pkt:
    ts: float = 0.0
    src: str = ""
    dst: str = ""
    proto: int = 0          # 6=TCP, 17=UDP
    sport: int = 0
    dport: int = 0
    ip_len: int = 0         # tamanho total do datagrama IP
    is_someip: bool = False
    is_sd: bool = False
    service_id: int = -1
    method_id: int = -1
    msg_type: int = -1
    client_id: int = -1
    # payloads (memoryview/bytes)
    pl_l4: bytes = b""      # payload de transporte (após TCP/UDP), inclui header SOME/IP
    pl_someip: bytes = b""  # payload da aplicação SOME/IP (após os 16 bytes), se não-SD
    pl_sd: bytes = b""      # payload SD (após os 16 bytes), se SD
    # entradas SD (apenas quando is_sd): lista de (type, service_id, instance_id, ttl)
    sd_entries: tuple = field(default_factory=tuple)


def parse(raw: bytes, ts: float = 0.0) -> Pkt | None:
    """Faz o parse de um frame Ethernet cru. Retorna Pkt ou None se não for IPv4/TCP/UDP."""
    if raw[12:14] != ETH_IPV4:
        return None
    ihl = (raw[14] & 0x0F) * 4
    proto = raw[23]
    ip_len = int.from_bytes(raw[16:18], "big")
    l4 = 14 + ihl
    p = Pkt(ts=ts, src=_ip(raw[26:30]), dst=_ip(raw[30:34]), proto=proto, ip_len=ip_len)

    if proto == 17:        # UDP
        p.sport = int.from_bytes(raw[l4:l4 + 2], "big")
        p.dport = int.from_bytes(raw[l4 + 2:l4 + 4], "big")
        off = l4 + 8
    elif proto == 6:       # TCP
        p.sport = int.from_bytes(raw[l4:l4 + 2], "big")
        p.dport = int.from_bytes(raw[l4 + 2:l4 + 4], "big")
        doff = (raw[l4 + 12] >> 4) * 4
        off = l4 + doff
    else:
        return None

    p.pl_l4 = raw[off:]
    if len(p.pl_l4) < 16:
        return p           # sem cabeçalho SOME/IP completo; ainda conta como TCP/UDP

    # cabeçalho SOME/IP
    p.service_id = int.from_bytes(p.pl_l4[0:2], "big")
    p.method_id = int.from_bytes(p.pl_l4[2:4], "big")
    p.client_id = int.from_bytes(p.pl_l4[8:10], "big")  # request_id = client_id(2)+session_id(2)
    p.msg_type = p.pl_l4[14]
    p.is_someip = True
    body = p.pl_l4[16:]

    if p.dport == SD_PORT or p.sport == SD_PORT or p.service_id == 0xFFFF:
        p.is_sd = True
        p.pl_sd = body
        p.sd_entries = _parse_sd_entries(body)
    else:
        p.pl_someip = body
    return p


def _parse_sd_entries(body: bytes) -> tuple:
    """Extrai entradas SD: tupla de (type, service_id, instance_id, ttl)."""
    if len(body) < 12:
        return ()
    try:
        len_entries = int.from_bytes(body[4:8], "big")
        start = 8
        end = start + len_entries
        entries = []
        i = start
        while i + 16 <= min(end, len(body)):
            etype = body[i]
            sid = int.from_bytes(body[i + 4:i + 6], "big")
            iid = int.from_bytes(body[i + 6:i + 8], "big")
            ttl = int.from_bytes(body[i + 9:i + 12], "big")
            entries.append((etype, sid, iid, ttl))
            i += 16
        return tuple(entries)
    except Exception:
        return ()


def flow_key(p: Pkt) -> tuple:
    """Chave de fluxo: mesmo par IP + portas (direcional, como no artigo)."""
    return (p.src, p.dst, p.sport, p.dport, p.proto)

## Feature engineering — 3/5 · Rotulador por assinatura de ataque

O tipo (multiclasse) vem do arquivo/cenário; a **assinatura por pacote** decide quais pacotes daquele
arquivo são de fato o ataque (o resto é benigno de fundo).

In [ ]:
NORMAL = 0


DOS = 1


FUZZY = 2


MITM_SINGLE = 3


MITM_MULTI = 4


NAMES = {NORMAL: "normal", DOS: "dos", FUZZY: "fuzzy",
         MITM_SINGLE: "mitm_single", MITM_MULTI: "mitm_multi"}


LEGIT_SERVICES = {0x1001, 0x1002, 0x1003}


ADAS_SERVICE = 0x1001


RELAY_SERVICE = 0x100B


def is_adas_notification(p: Pkt) -> bool:
    return (p.service_id == ADAS_SERVICE and p.method_id == 0x0001
            and p.msg_type == MT_NOTIFICATION)


def _has_withdraw(p: Pkt) -> bool:
    """SD OfferService com TTL=0 para serviço legítimo (stopOffer / spoofing)."""
    for (etype, sid, iid, ttl) in p.sd_entries:
        if etype == SD_OFFER and ttl == 0 and sid in LEGIT_SERVICES:
            return True
    return False


def _has_fuzzy_offer(p: Pkt) -> bool:
    """SD OfferService anunciando serviço fora do conjunto legítimo."""
    for (etype, sid, iid, ttl) in p.sd_entries:
        if etype == SD_OFFER and ttl != 0 and sid not in LEGIT_SERVICES:
            return True
    return False


def sig_fuzzy(p: Pkt) -> bool:
    return _has_fuzzy_offer(p)


def sig_mitm_single(p: Pkt) -> bool:
    return _has_withdraw(p)


def sig_mitm_multi(p: Pkt) -> bool:
    return p.service_id == RELAY_SERVICE or _has_withdraw(p)


STATIC_SIG = {
    FUZZY: sig_fuzzy,
    MITM_SINGLE: sig_mitm_single,
    MITM_MULTI: sig_mitm_multi,
}

## Feature engineering — 4/5 · Ajuste do modelo de bytes (benigno) + constantes

In [ ]:
L = {"someip": 64, "sd": 80, "l4": 64}


PCAP_DIR_DEFAULT = "data/pcap"


ATTACK_FILES = {
    "dos_noti_flood.pcap": DOS,
    "fuzzy_sd_offer_rand_noti(1).pcap": FUZZY,
    "fuzzy_sd_offer_rand_noti(2).pcap": FUZZY,
    "fuzzy_sd_offer_rand_noti(3).pcap": FUZZY,
    "mitm_multi_attacker.pcap": MITM_MULTI,
    "mitm_single_attacker.pcap": MITM_SINGLE,
}


BENIGN_FILE = "benign_traffic.pcap"


def _ts(meta) -> float:
    try:
        return meta.sec + meta.usec * 1e-6
    except Exception:
        try:
            return meta[0] + meta[1] * 1e-6
        except Exception:
            return 0.0


def fit_bytemodels(pcap_dir: str, alpha: float = 1.0, max_packets: int | None = None):
    """Treina os 3 modelos de bytes no tráfego benigno e coleta o conjunto de IPs de
    origem benignos (usado para identificar o nó atacante em cada cenário).
    Retorna (models, benign_src_ips)."""
    models = {k: ByteModel(L[k], alpha) for k in L}
    benign_ips = set()
    n = 0
    for raw, meta in RawPcapReader(os.path.join(pcap_dir, BENIGN_FILE)):
        n += 1
        if max_packets and n > max_packets:
            break
        p = parse(raw)
        if p is None:
            continue
        benign_ips.add(p.src)
        if p.pl_l4:
            models["l4"].update(p.pl_l4)
        if p.is_sd and p.pl_sd:
            models["sd"].update(p.pl_sd)
        elif p.pl_someip:
            models["someip"].update(p.pl_someip)
    for m in models.values():
        m.finalize()
    return models, benign_ips

## Feature engineering — 5/5 · Extração das features por pacote

`extract_file` varre um PCAP mantendo estado **por fluxo** e **por origem**, e produz as 21 colunas
(usamos 16). As 4 comportamentais: `repeat_rate` (janela de 5 por fluxo), `someip_len`, `l4_len`,
`src_payload_div` (janela de 1000 por origem).

In [ ]:
RELAY_SERVICE = 0x100B


FEATURE_NAMES = [
    # core (12)
    "ip_time_interval", "someip_loglik", "sd_loglik", "l4_loglik",
    "someip_entropy", "sd_entropy", "l4_entropy",
    "someip_change", "sd_change", "l4_change", "ip_len_change", "l4_len_change",
    # ext (9)
    "repeat_rate", "someip_len", "l4_len", "src_pkt_rate", "src_payload_div",
    "is_sd", "src_service_div", "is_relay", "src_clientid_div",
]


def extract_file(path, models, attack_type=NORMAL, benign_ips=None, max_packets=None):
    last = {}
    benign_ips = benign_ips or set()
    attacker_ips = set()
    # estado das features comportamentais
    recent_pld = defaultdict(lambda: deque(maxlen=5))      # por fluxo
    src_ts = defaultdict(lambda: deque(maxlen=1000))       # por src
    src_pld = defaultdict(lambda: deque(maxlen=1000))      # por src
    src_svc = defaultdict(lambda: deque(maxlen=100))       # por src
    src_cid = defaultdict(lambda: deque(maxlen=100))       # por src
    X, ymul = [], []
    n = 0
    for raw, meta in RawPcapReader(path):
        n += 1
        if max_packets and n > max_packets:
            break
        p = parse(raw, _ts(meta))
        if p is None:
            continue
        fk = flow_key(p)
        st = last.get(fk)

        # --- core relacionais ---
        if st is None:
            iat = 0.0; ip_len_chg = 0; l4_len_chg = 0
            someip_chg = sd_chg = l4_chg = 0
        else:
            iat = p.ts - st["ts"]
            ip_len_chg = p.ip_len - st["ip_len"]
            l4_len_chg = len(p.pl_l4) - len(st["l4"])
            l4_chg = hamming(p.pl_l4, st["l4"])
            someip_chg = hamming(p.pl_someip, st["someip"]) if (p.pl_someip or st["someip"]) else 0
            sd_chg = hamming(p.pl_sd, st["sd"]) if (p.pl_sd or st["sd"]) else 0

        someip_ll = models["someip"].loglik(p.pl_someip) if p.pl_someip else 0.0
        sd_ll = models["sd"].loglik(p.pl_sd) if p.pl_sd else 0.0
        l4_ll = models["l4"].loglik(p.pl_l4) if p.pl_l4 else 0.0
        someip_h = models["someip"].cross_entropy(p.pl_someip) if p.pl_someip else 0.0
        sd_h = models["sd"].cross_entropy(p.pl_sd) if p.pl_sd else 0.0
        l4_h = models["l4"].cross_entropy(p.pl_l4) if p.pl_l4 else 0.0

        # --- ext comportamentais ---
        # f13 repeat rate (fração dos últimos 5 payloads SOME/IP iguais ao atual)
        h = recent_pld[fk]
        repeat_rate = (sum(1 for q in h if q == p.pl_someip) / len(h)) if (h and p.pl_someip) else 0.0
        h.append(p.pl_someip)
        # f15/f16 comprimentos
        someip_len = float(len(p.pl_someip))
        l4_len = float(len(p.pl_l4))
        # f17 taxa de pacotes por src (janela 1000)
        wt = src_ts[p.src]; wt.append(p.ts)
        if len(wt) >= 2:
            d = wt[-1] - wt[0]
            src_pkt_rate = (len(wt) - 1) / d if d > 0 else float(len(wt))
        else:
            src_pkt_rate = 0.0
        # f18 diversidade de payload por src
        wp = src_pld[p.src]
        if p.pl_someip:
            wp.append(p.pl_someip)
        src_payload_div = (len(set(wp)) / len(wp)) if len(wp) > 1 else 0.0
        # f19 is_sd
        is_sd = 1.0 if p.is_sd else 0.0
        # f20 diversidade de serviços por src
        ws = src_svc[p.src]
        if p.service_id >= 0:
            ws.append(p.service_id)
        src_service_div = float(len(set(ws))) if ws else 1.0
        # f21 relay service
        is_relay = 1.0 if p.service_id == RELAY_SERVICE else 0.0
        # f22 diversidade de client_id em REQUESTs (msg_type < 0x80) por src
        wc = src_cid[p.src]
        if p.client_id >= 0 and 0 <= p.msg_type < 0x80:
            wc.append(p.client_id)
        src_clientid_div = float(len(set(wc))) if wc else 1.0

        X.append((iat, someip_ll, sd_ll, l4_ll, someip_h, sd_h, l4_h,
                  someip_chg, sd_chg, l4_chg, ip_len_chg, l4_len_chg,
                  repeat_rate, someip_len, l4_len, src_pkt_rate, src_payload_div,
                  is_sd, src_service_div, is_relay, src_clientid_div))

        atk = NORMAL
        if attack_type != NORMAL and p.src not in benign_ips:
            atk = attack_type
            attacker_ips.add(p.src)
        ymul.append(atk)

        last[fk] = {"ts": p.ts, "ip_len": p.ip_len, "l4": p.pl_l4,
                    "someip": p.pl_someip, "sd": p.pl_sd}
    return X, ymul, attacker_ips

## Pipeline: download → extração → treino → avaliação (split temporal)

In [ ]:
# Baixa os PCAPs crus (~1.9GB, figshare)
if not os.path.exists(os.path.join(PCAP_DIR_DEFAULT, BENIGN_FILE)):
    print('baixando PCAPs (~1.9GB)...'); os.system(f'python scripts/download_pcaps.py')
print('PCAPs:', sorted(os.listdir(PCAP_DIR_DEFAULT)) if os.path.isdir(PCAP_DIR_DEFAULT) else 'AUSENTE')

In [ ]:
# EXTRACAO ponta a ponta: PCAP cru -> features (cronometrada)
t0 = time.perf_counter(); models, benign_ips = fit_bytemodels(PCAP_DIR_DEFAULT); t_fit = time.perf_counter() - t0
print(f'modelos de bytes (benigno): {t_fit:.1f}s | IPs benignos: {len(benign_ips)}')
files = [(BENIGN_FILE, NORMAL)] + list(ATTACK_FILES.items())
Xparts, yparts, counts, t_ext = [], [], [], 0.0
for fname, atk in files:
    path = os.path.join(PCAP_DIR_DEFAULT, fname)
    t0 = time.perf_counter(); Xf, ymf, _ = extract_file(path, models, atk, benign_ips); t_ext += time.perf_counter() - t0
    Xparts.append(np.asarray(Xf, dtype=np.float64)); yparts.append(np.asarray(ymf, dtype=np.int64))
    counts.append(len(ymf)); print(f'  {fname:38} pkts={len(ymf):>9,}')
Xraw = np.concatenate(Xparts); y = np.concatenate(yparts); del Xparts, yparts
xmin = Xraw.min(0); xmax = Xraw.max(0); rng = np.where(xmax > xmin, xmax - xmin, 1.0)   # min-max por coluna
Xn = ((Xraw - xmin) / rng).astype(np.float32); del Xraw
n_ext = y.size; ext_us = t_ext / n_ext * 1e6
print(f'\nEXTRACAO: {n_ext:,} pacotes em {t_ext:.1f}s -> {ext_us:.2f} us/pacote  ({n_ext/t_ext:,.0f} pkt/s)')

In [ ]:
# Seleciona as 16 features comportamentais e faz o split temporal por arquivo
import matplotlib.pyplot as plt
from sklearn.metrics import (classification_report, confusion_matrix, roc_curve, auc,
                             precision_recall_curve, average_precision_score)
from sklearn.preprocessing import label_binarize
from xgboost import XGBClassifier

CLASSES = ['normal', 'dos', 'fuzzy', 'mitm_single', 'mitm_multi']; N = len(CLASSES)
RESULTS = 'results'; os.makedirs(RESULTS, exist_ok=True)
CONTENT_EXT = list(range(12)) + [12, 13, 14, 16]                 # 16 features, sem cabecalho
PARAMS = dict(objective='multi:softprob', num_class=N, n_estimators=300, max_depth=8,
              learning_rate=0.3, tree_method='hist', device='cuda',  # sem GPU: device='cpu'
              max_bin=256, n_jobs=-1, eval_metric='mlogloss')
X16 = Xn[:, CONTENT_EXT]; FNAMES = [FEATURE_NAMES[i] for i in CONTENT_EXT]
print('16 features:', FNAMES)

def temporal_split(counts, frac=0.70):
    tr, te, s = [], [], 0
    for c in counts:
        cut = s + int(c * frac); tr.append(np.arange(s, cut)); te.append(np.arange(cut, s + c)); s += c
    return np.concatenate(tr), np.concatenate(te)
itr, ite = temporal_split(counts); y_te = y[ite]
print(f'treino = {itr.size:,}   teste = {ite.size:,}')
print('teste por classe:', dict(zip(CLASSES, np.bincount(y_te, minlength=N).tolist())))

In [ ]:
# Modelo: 16 features comportamentais
clf = XGBClassifier(**PARAMS); clf.fit(X16[itr], y[itr])
proba = clf.predict_proba(X16[ite]); y_pred = proba.argmax(1)
rep = classification_report(y_te, y_pred, target_names=CLASSES, digits=4, output_dict=True)
macro_f1 = rep['macro avg']['f1-score']; acc = rep['accuracy']
print(classification_report(y_te, y_pred, target_names=CLASSES, digits=4))
print(f'macro-F1 = {macro_f1:.4f}   accuracy = {acc:.4f}')

In [ ]:
# Matriz de confusao (normalizada por linha) -> results/cm_temporal.png
cm = confusion_matrix(y_te, y_pred); cmn = cm / cm.sum(1, keepdims=True)
fig, ax = plt.subplots(figsize=(6.5, 5.5)); im = ax.imshow(cmn, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(N)); ax.set_yticks(range(N))
ax.set_xticklabels(CLASSES, rotation=45, ha='right'); ax.set_yticklabels(CLASSES)
ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title('Confusion matrix (temporal split)')
for i in range(N):
    for j in range(N):
        ax.text(j, i, f'{cmn[i,j]*100:.1f}%\n({cm[i,j]:,})', ha='center', va='center',
                fontsize=7, color='white' if cmn[i,j] > 0.5 else 'black')
fig.colorbar(im, fraction=0.046, pad=0.04); plt.tight_layout()
fig.savefig(f'{RESULTS}/cm_temporal.png', dpi=200, bbox_inches='tight'); plt.show()

In [ ]:
# Curvas ROC e Precisao-Recall (one-vs-rest) -> results/roc_pr_temporal.png
yb = label_binarize(y_te, classes=range(N))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5)); roc_auc = {}; pr_ap = {}
for i, name in enumerate(CLASSES):
    fpr, tpr, _ = roc_curve(yb[:, i], proba[:, i]); a = auc(fpr, tpr); roc_auc[name] = float(a)
    s = max(len(fpr) // 2000, 1); ax1.plot(fpr[::s], tpr[::s], lw=1.6, label=f'{name} (AUC={a:.3f})')
    prec, rec, _ = precision_recall_curve(yb[:, i], proba[:, i])
    ap = average_precision_score(yb[:, i], proba[:, i]); pr_ap[name] = float(ap)
    s2 = max(len(rec) // 2000, 1); ax2.plot(rec[::s2], prec[::s2], lw=1.6, label=f'{name} (AP={ap:.3f})')
ax1.plot([0, 1], [0, 1], 'k--', lw=0.7); ax1.set_xlabel('False Positive Rate'); ax1.set_ylabel('True Positive Rate')
ax1.set_title('ROC (one-vs-rest, temporal)'); ax1.legend(fontsize=8, loc='lower right')
ax2.set_xlabel('Recall'); ax2.set_ylabel('Precision'); ax2.set_title('Precision-Recall (one-vs-rest, temporal)')
ax2.legend(fontsize=8, loc='lower left'); plt.tight_layout()
fig.savefig(f'{RESULTS}/roc_pr_temporal.png', dpi=200, bbox_inches='tight'); plt.show()

In [ ]:
# Importancia das features (ganho) -> results/feature_importance.png
score = clf.get_booster().get_score(importance_type='gain')
imp = np.array([score.get(f'f{i}', 0.0) for i in range(len(CONTENT_EXT))])
order = np.argsort(imp)
fig, ax = plt.subplots(figsize=(7, 5.5)); ax.barh([FNAMES[i] for i in order], imp[order])
ax.set_xlabel('Importancia (gain, XGBoost)'); ax.set_title('Importancia das 16 features (split temporal)')
plt.tight_layout(); fig.savefig(f'{RESULTS}/feature_importance.png', dpi=200, bbox_inches='tight'); plt.show()
feat_imp = {FNAMES[i]: float(imp[i]) for i in order[::-1]}
for k, v in feat_imp.items(): print(f'  {k:18} {v:10.2f}')

In [ ]:
# Tempo de INFERENCIA (por pacote)
Xte = X16[ite]; _ = clf.predict_proba(Xte[:2000])              # warm-up
t0 = time.perf_counter(); _ = clf.predict_proba(Xte); inf_total = time.perf_counter() - t0
inf_us = inf_total / Xte.shape[0] * 1e6
print(f'Inferencia ({PARAMS["device"]}): {Xte.shape[0]:,} pacotes em {inf_total:.3f}s '
      f'-> {inf_us:.2f} us/pacote  ({Xte.shape[0]/inf_total:,.0f} pkt/s)')
print(f'Razao extracao/inferencia ~ {ext_us/inf_us:.1f}x  (a extracao domina o custo por pacote)')

In [ ]:
# Ablacao: 12 features (nucleo, sem as 4 adicionais) vs as 16, mesmo split temporal
clf12 = XGBClassifier(**PARAMS); clf12.fit(X16[itr][:, :12], y[itr])
pred12 = clf12.predict(X16[ite][:, :12])
rep12 = classification_report(y_te, pred12, target_names=CLASSES, digits=4, output_dict=True)
macro12 = rep12['macro avg']['f1-score']
print(f'Ablacao (temporal) macro-F1:  12 features = {macro12:.4f}   |   16 features = {macro_f1:.4f}')
for c in CLASSES:
    print(f'  {c:12}  12={rep12[c]["f1-score"]:.4f}   16={rep[c]["f1-score"]:.4f}')

In [ ]:
# Salva todas as metricas -> results/metrics_temporal.json
out = {
    'machine': MACHINE, 'split': 'temporal_by_file_70_30', 'params': dict(PARAMS),
    'test_counts': dict(zip(CLASSES, np.bincount(y_te, minlength=N).tolist())),
    'model_16feat': {'macro_f1': float(macro_f1), 'accuracy': float(acc),
        'per_class': {c: {k: float(rep[c][k]) for k in ('precision', 'recall', 'f1-score')} for c in CLASSES},
        'roc_auc': roc_auc, 'pr_ap': pr_ap, 'feature_importance_gain': feat_imp},
    'ablation_12feat': {'macro_f1': float(macro12), 'per_class_f1': {c: float(rep12[c]['f1-score']) for c in CLASSES}},
    'timing': {'extraction_us_per_pkt': float(ext_us), 'inference_us_per_pkt': float(inf_us),
               'extraction_over_inference': float(ext_us / inf_us),
               'bytemodel_fit_s': float(t_fit), 'extraction_total_s': float(t_ext)},
}
json.dump(out, open(f'{RESULTS}/metrics_temporal.json', 'w'), indent=2)
print('salvo:', os.path.abspath(RESULTS))
print(json.dumps({'macro_f1_16': macro_f1, 'macro_f1_12': macro12, 'accuracy': acc,
                  'ext_us': round(ext_us, 2), 'inf_us': round(inf_us, 2)}, indent=2))

## Leitura

- **Ponta a ponta e transparente:** os resultados saem do **tráfego cru**, com toda a extração de features
  visível acima (byte model, parser, rotulador, 16 features).
- **macro-F1 (temporal) ~0,97**, **recall macro ~0,996**: o detector praticamente não deixa ataque passar.
- **`mitm_multi`** é a única classe que cai — por **precisão** (falso alarme), não por *recall*: o MITM *relay*
  **republica notificação legítima**, a fronteira mais borrada frente ao `normal`.
- **Ablação:** com **12 features** o macro-F1 desaba (confusão em `fuzzy` e `mitm_multi`); as **4 adicionais**
  recuperam essas classes — o gargalo era a **representação**, não o modelo.
- **Custo:** a **extração domina** o tempo por pacote frente à **inferência**.

Artefatos em `results/`: `cm_temporal.png`, `roc_pr_temporal.png`, `feature_importance.png`, `metrics_temporal.json`.